## Desafío 3

##### Configuración inicial

In [183]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import matplotlib.pyplot as plt
from keras.src.callbacks import EarlyStopping
from tensorflow.keras.callbacks import Callback
from keras.src.layers import GRU
from keras import Input
from tensorflow.keras.regularizers import l2


### Datos
Utilizaré como dataset letras de temas de Charly Garcia

In [184]:
df = pd.read_csv('datasets/letras_charly.csv', usecols=['verso']).dropna().drop_duplicates()

print("Cantidad de versos:", df.shape[0])
df.head()

Cantidad de versos: 4826


,verso
0,Podés pasear en limousine
1,cortar las flores del jardín
2,podés cambiar el sol
3,y esconderte si no quieres verme.
4,Puedes ver amanecer


In [185]:
vocabulario = {' ': 1, 'e': 2, 'a': 3, 'o': 4, 's': 5, 'n': 6, 'r': 7, 'i': 8, 'l': 9, 't': 10, 'u': 11, 'd': 12, 'm': 13,
               'c': 14, 'p': 15, 'y': 16, 'v': 17, 'h': 18, 'q': 19, 'g': 20, 'b': 21, ',': 22, '.': 23, 'f': 24, 'á': 25,
               'í': 26, 'ó': 27, 'z': 28, 'é': 29, 'j': 30, 'w': 31, 'k': 32, 'ñ': 35, '?': 36, 'x': 37, 'ú': 38, '!': 39,
               '"': 40, '¿': 41, "'": 42, ')': 43, '(': 44, '’': 45, ':': 46, '-': 49, '¡': 50, '/': 59, 'ü': 60}


def limpiar_texto(texto):
    caracteres_validos = set(vocabulario.keys())
    texto_limpio = ''.join([c for c in texto.lower() if c in caracteres_validos])
    return texto_limpio

In [186]:
corpus = []

for _, row in df.iterrows():
    verso_limpio = limpiar_texto(row.iloc[0])  # Limpiar caracteres no válidos
    palabras = text_to_word_sequence(verso_limpio)  # Tokenizar en palabras válidas

    if not palabras:
        continue  # saltar si quedó vacío

    verso = ' '.join(palabras) + ' '  # reconstruir verso con espacios y uno final
    verso_lista = list(verso)         # convertir el string en lista de caracteres

    corpus.append(verso_lista)

print(f"El corpus posee {len(corpus)} documentos (versos). \nPor ejemplo: {corpus[0:2]}")


El corpus posee 4825 documentos (versos). 
Por ejemplo: [['p', 'o', 'd', 'é', 's', ' ', 'p', 'a', 's', 'e', 'a', 'r', ' ', 'e', 'n', ' ', 'l', 'i', 'm', 'o', 'u', 's', 'i', 'n', 'e', ' '], ['c', 'o', 'r', 't', 'a', 'r', ' ', 'l', 'a', 's', ' ', 'f', 'l', 'o', 'r', 'e', 's', ' ', 'd', 'e', 'l', ' ', 'j', 'a', 'r', 'd', 'í', 'n', ' ']]


In [187]:
length_secuence = [len(secuence) for secuence in corpus]
plt.hist(length_secuence,bins=10)

(array([7.430e+02, 2.749e+03, 1.196e+03, 1.160e+02, 1.100e+01, 7.000e+00,
        1.000e+00, 0.000e+00, 1.000e+00, 1.000e+00]),
 array([  3. ,  16.7,  30.4,  44.1,  57.8,  71.5,  85.2,  98.9, 112.6,
        126.3, 140. ]),
 <BarContainer object of 10 artists>)

Ahora tokenizaré cada una de las letras

In [188]:
max_context_size = 70

In [189]:
tok = Tokenizer()
tok.fit_on_texts(corpus)
tokenized_corpus = tok.texts_to_sequences(corpus)

print(tokenized_corpus[0:9])

[[15, 4, 11, 26, 5, 1, 15, 3, 5, 2, 3, 7, 1, 2, 6, 1, 9, 8, 13, 4, 10, 5, 8, 6, 2, 1], [14, 4, 7, 12, 3, 7, 1, 9, 3, 5, 1, 24, 9, 4, 7, 2, 5, 1, 11, 2, 9, 1, 28, 3, 7, 11, 23, 6, 1], [15, 4, 11, 26, 5, 1, 14, 3, 13, 19, 8, 3, 7, 1, 2, 9, 1, 5, 4, 9, 1], [16, 1, 2, 5, 14, 4, 6, 11, 2, 7, 12, 2, 1, 5, 8, 1, 6, 4, 1, 17, 10, 8, 2, 7, 2, 5, 1, 18, 2, 7, 13, 2, 1], [15, 10, 2, 11, 2, 5, 1, 18, 2, 7, 1, 3, 13, 3, 6, 2, 14, 2, 7, 1], [14, 4, 6, 1, 14, 3, 18, 8, 3, 7, 1, 11, 2, 5, 11, 2, 1, 10, 6, 1, 21, 4, 12, 2, 9, 1], [16, 1, 6, 4, 1, 12, 8, 2, 6, 2, 5, 1, 10, 6, 1, 15, 4, 17, 10, 8, 12, 4, 1, 11, 2, 1, 3, 13, 4, 7, 1, 15, 3, 7, 3, 1, 11, 3, 7, 1], [16, 2, 6, 11, 4, 1, 11, 2, 1, 9, 3, 1, 14, 3, 13, 3, 1, 3, 9, 1, 9, 8, 18, 8, 6, 20, 1], [5, 8, 2, 6, 12, 2, 5, 1, 2, 9, 1, 2, 6, 14, 8, 2, 7, 7, 4, 1]]


Separamos sets de entrenamiento y validación

In [190]:
train, val, _, _ = train_test_split(tokenized_corpus, tokenized_corpus, test_size=0.2, random_state=42)
train


[[16, 1, 7, 4, 19, 2, 6, 1, 9, 4, 1, 17, 10, 2, 1, 16, 4, 1, 20, 3, 6, 26, 1],
 [15, 3, 7, 3, 1, 10, 6, 1, 22, 6, 20, 2, 9, 1],
 [9,
  3,
  1,
  3,
  9,
  2,
  20,
  7,
  23,
  3,
  1,
  6,
  4,
  1,
  2,
  5,
  1,
  5,
  4,
  9,
  4,
  1,
  19,
  7,
  3,
  5,
  8,
  9,
  2,
  7,
  3,
  1],
 [6, 3, 14, 8, 25, 1, 2, 6, 1, 9, 3, 1, 8, 6, 20, 7, 3, 18, 8, 11, 2, 27, 1],
 [11,
  2,
  1,
  9,
  4,
  5,
  1,
  14,
  4,
  6,
  5,
  2,
  28,
  4,
  5,
  1,
  17,
  10,
  2,
  1,
  12,
  2,
  1,
  11,
  3,
  6,
  1],
 [2, 6, 1, 10, 6, 1, 5, 10, 19, 12, 2, 7, 7, 22, 6, 2, 4, 1],
 [5, 25, 9, 4, 1, 18, 2, 7, 13, 2, 1, 2, 6, 1, 10, 6, 1, 15, 3, 15, 2, 9, 1],
 [14, 7, 2, 4, 1, 17, 10, 2, 1, 2, 6, 12, 2, 6, 11, 2, 7, 22, 5, 1],
 [2,
  6,
  1,
  15,
  9,
  2,
  6,
  3,
  1,
  9,
  9,
  10,
  18,
  8,
  3,
  1,
  6,
  4,
  1,
  5,
  2,
  1,
  18,
  3,
  1,
  3,
  1,
  13,
  4,
  28,
  3,
  7,
  1],
 [9,
  3,
  1,
  6,
  4,
  5,
  12,
  3,
  9,
  20,
  8,
  3,
  1,
  17,
  10,
  2,
  1,
  4,
  12,
  7,
 

Realizamos un padding para ajustar al máximo valor de contexto

In [191]:
def seq_of(set, data_aug=True):
    train_seq = []

    for word in set:
        if data_aug:
            subseq =  [word[:i+2] for i in range(len(word)-1)]
        else:
            subseq = [word]
        train_seq.append(pad_sequences(subseq, maxlen=max_context_size+1, padding='pre'))
    train_seqs = np.concatenate(train_seq, axis=0)

    return train_seqs, train_seqs[:, :-1], train_seqs[:, -1]

In [201]:
train_seqs, X, y = seq_of(train)
print(f'training shape {train_seqs.shape}')
print(f'X shape {X.shape}')
print(f'y length {len(y)}')
print(f'\n')
for i in range(100):
    print(f'(X): {X[i]} ({ "".join([ tok.index_word.get(c,"") for c in X[i]]) }) (y): {y[i]} ({tok.index_word[y[i]]})')

training shape (94998, 71)
X shape (94998, 70)
y length 94998


(X): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 16] (y) (y): 1 ( )
(X): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 16  1] (y ) (y): 7 (r)
(X): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 16  1  7] (y r) (y): 4 (o)
(X): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 16  1  7  4] (y ro) (y): 

In [193]:
vocab_size = len(tok.word_counts)
print(f'Tamaño del vocabulario: {vocab_size} \n')
print(f'Vocabulario: {tok.word_index} \n')
print(f'Vocabulario por apariciones: {tok.word_docs}')

Tamaño del vocabulario: 36 

Vocabulario: {' ': 1, 'e': 2, 'a': 3, 'o': 4, 's': 5, 'n': 6, 'r': 7, 'i': 8, 'l': 9, 'u': 10, 'd': 11, 't': 12, 'm': 13, 'c': 14, 'p': 15, 'y': 16, 'q': 17, 'v': 18, 'b': 19, 'g': 20, 'h': 21, 'á': 22, 'í': 23, 'f': 24, 'ó': 25, 'é': 26, 'z': 27, 'j': 28, 'ñ': 29, 'x': 30, 'ú': 31, 'k': 32, '¿': 33, 'w': 34, '¡': 35, 'ü': 36} 

Vocabulario por apariciones: defaultdict(<class 'int'>, {'d': 2782, 'é': 445, 'e': 4453, 'i': 3142, 'o': 4032, 'r': 3628, 'm': 2437, 'p': 1903, ' ': 4825, 'a': 4224, 'l': 2925, 'u': 3090, 'n': 3735, 's': 3689, 'f': 524, 'c': 2293, 'j': 427, 'í': 590, 't': 2895, 'b': 994, 'y': 1484, 'q': 1409, 'v': 1287, 'h': 875, 'g': 1020, 'ó': 480, 'ñ': 147, 'z': 449, 'á': 611, 'ú': 71, 'x': 76, 'w': 21, 'k': 49, '¿': 41, '¡': 12, 'ü': 2})


### Modelo

In [205]:
model = Sequential()
model.add(Input(shape=(max_context_size,)))
model.add(Embedding(input_dim=vocab_size+1, output_dim=64)) 
model.add(GRU(128, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(64, kernel_regularizer=l2(1e-4), activation='relu'))
model.add(Dense(vocab_size+1, activation='softmax'))

model.compile(loss=SparseCategoricalCrossentropy(), optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_12 (Embedding)        │ (None, 70, 64)         │         2,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 128)            │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 37)             │         2,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 87,525 (341.89 KB)

 Trainable params: 87,525 (341.89 KB)

 Non-trainable params: 0 (0.00 B)

In [206]:
class PplCallback(Callback):
    def __init__(self, val_data, max_context_size):
        super().__init__()
        self.max_context_size = max_context_size
        self.X_val, self.y_val = self._preparar_datos(val_data)

    def _preparar_datos(self, corpus):
        X = []
        y = []

        for seq in corpus:
            for i in range(1, len(seq)):
                contexto = seq[max(0, i - self.max_context_size):i]
                target = seq[i]

                contexto_ids = [c for c in contexto]
                contexto_ids = [0] * (self.max_context_size - len(contexto_ids)) + contexto_ids

                X.append(contexto_ids)
                y.append(target)

        return np.array(X), np.array(y)

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}

        val_loss, val_acc = self.model.evaluate(self.X_val, self.y_val, verbose=0)
        val_ppl = np.exp(val_loss)

        logs['val_loss'] = val_loss
        logs['val_accuracy'] = val_acc
        logs['val_perplexity'] = val_ppl

        print(f'\n📏 Validation loss: {val_loss:.4f} | accuracy: {val_acc:.4f} | perplexity: {val_ppl:.4f}')


In [207]:
_, X_val, y_val =seq_of(val)

ppl_callback = PplCallback(val_data=val, max_context_size=max_context_size)
early_stopping = EarlyStopping(monitor='val_perplexity', patience=5, restore_best_weights=True, mode='min')

hist = model.fit(X, y, epochs=20, callbacks=[ppl_callback, early_stopping], batch_size=256)

Epoch 1/20
371/372 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.2612 - loss: 2.6803
📏 Validation loss: 2.0398 | accuracy: 0.3663 | perplexity: 7.6894
372/372 ━━━━━━━━━━━━━━━━━━━━ 63s 166ms/step - accuracy: 0.2615 - loss: 2.6785 - val_loss: 2.0398 - val_accuracy: 0.3663 - val_perplexity: 7.6894
Epoch 2/20
371/372 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.3716 - loss: 2.0297
📏 Validation loss: 1.8886 | accuracy: 0.4045 | perplexity: 6.6100
372/372 ━━━━━━━━━━━━━━━━━━━━ 62s 167ms/step - accuracy: 0.3717 - loss: 2.0294 - val_loss: 1.8886 - val_accuracy: 0.4045 - val_perplexity: 6.6100
Epoch 3/20
371/372 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.4105 - loss: 1.8906
📏 Validation loss: 1.7799 | accuracy: 0.4434 | perplexity: 5.9292
372/372 ━━━━━━━━━━━━━━━━━━━━ 52s 139ms/step - accuracy: 0.4106 - loss: 1.8905 - val_loss: 1.7799 - val_accuracy: 0.4434 - val_perplexity: 5.9292
Epoch 4/20
371/372 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.4391 - loss: 1.7946
📏 Validation lo

In [208]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def generate_seq(model, tokenizer, seed_text, max_length, n_chars=100, temperature=1.0):
    result = list(seed_text.lower())

    for _ in range(n_chars):
        # Convertir contexto a índices
        encoded = [tokenizer.word_index.get(c, 0) for c in result[-max_length:]]
        encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')

        # Obtener predicciones
        preds = model.predict(encoded, verbose=0)[0]

        # Aplicar temperatura
        preds = np.log(preds + 1e-10) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)

        # Elegir siguiente carácter con muestreo probabilístico
        next_index = np.random.choice(len(preds), p=preds)
        next_char = tokenizer.index_word.get(next_index, '')

        result.append(next_char)

    return ''.join(result)


In [213]:
input_text='los dinosaurios'

print(generate_seq(model, tok, input_text, max_length=max_context_size, n_chars=100, temperature=0.05))

los dinosaurios que están para desaparecer no me digar el mundo de amor es tan disoria de la cama al fantas de la c


Si bien el modelo crea palabras sintacticamente correctas, no logré que el modelo pudiera armar oraciones que tuvieran sentido. De todos modos creo que es un buen resultado, sobre todo pensando que logra predecir correctamente los espacios en blanco entre palabras. Es decir, logra representar separaciones de estas.